In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import random
import matplotlib
import warnings

from scipy.interpolate import interp1d
from matplotlib.colors import to_rgb, LinearSegmentedColormap
from scipy.stats import zscore


### Read data

In [3]:
### Without cell cycle information
data_1 = pd.read_csv("../data/Su_2020/chromosome21.tsv", sep='\t')
data_1['Chr'] = data_1['Genomic coordinate'].apply(lambda x: x.split(':')[0][3:])
data_1['Coord1'] = data_1['Genomic coordinate'].apply(lambda x: int(x.split(':')[1].split('-')[0]))
data_1['Coord2'] = data_1['Genomic coordinate'].apply(lambda x: int(x.split(':')[1].split('-')[1]))
data_1.drop(['Genomic coordinate', 'TSS ZXY(nm)'], inplace=True, axis=1)

### With cell cycle information
data_2 = pd.read_csv("../data/Su_2020/chromosome21-cell_cycle.tsv", sep='\t')
data_2['Chr'] = data_2['Genomic coordinate'].apply(lambda x: x.split(':')[0][3:])
data_2['Coord1'] = data_2['Genomic coordinate'].apply(lambda x: int(x.split(':')[1].split('-')[0]))
data_2['Coord2'] = data_2['Genomic coordinate'].apply(lambda x: int(x.split(':')[1].split('-')[1]))
data_2.drop(['Genomic coordinate', 'TSS ZXY(nm)', 'Cell cycle state'], inplace=True, axis=1)

# Change trace id
data_2['Chromosome copy number'] = data_2['Chromosome copy number']  + data_1['Chromosome copy number'].max()

# Join datasets
MERFISH_data = pd.concat([data_1, data_2], axis = 0)

In [ ]:
### Filter region
start = 28000001
end = 29910001

res = 50000
n_steps = int((end - start)/res)

eff_thr = 0.95
#consec_dist_thr = 600 # Remove beads located at more than 600 nm from neighbouring beads
consec_dist_thr = 1000 # Remove beads located at more than 1000 nm from neighbouring beads - increases dataset size

MERFISH_data = MERFISH_data[(MERFISH_data['Coord1'] >= start) & (MERFISH_data['Coord2'] <= end)]
MERFISH_data = MERFISH_data.copy()
MERFISH_data['barcode'] = [int(num) for num in (MERFISH_data['Coord1'] - start) / res + 1]

n_traces = len(MERFISH_data['Chromosome copy number'].unique())

In [8]:
# Extract transcriptional data (target value in model)
MERFISH_data_transcription = MERFISH_data[~MERFISH_data['Transcription'].isna()].iloc[:, 3:6]
MERFISH_data_transcription['Gene names'] = MERFISH_data_transcription['Gene names'].apply(lambda x: x[:-1])
MERFISH_data_transcription = MERFISH_data_transcription.pivot(index = 'Chromosome copy number', columns='Gene names', values ='Transcription')
MERFISH_data_transcription.columns.name = None
MERFISH_data_transcription.reset_index(inplace=True)

# Remove transcriptional data from positional data
MERFISH_data.drop(['Gene names', 'Transcription'], inplace=True, axis=1)

#### Filter low efficiency barcodes

In [9]:
# Find barcodes with efficiency < 10 %
MERFISH_data['missing'] = np.isnan(MERFISH_data['Z(nm)'])
n_missing = pd.DataFrame(MERFISH_data.groupby('barcode').sum()['missing'])
n_missing['perc'] = n_missing['missing'] / n_traces * 100
bad_barcodes = n_missing[n_missing['perc'] > 90].index.tolist()

# Remove barcodes with low efficiency
MERFISH_data_filtered = MERFISH_data.copy()
for barcode in bad_barcodes:
    MERFISH_data_filtered.loc[MERFISH_data_filtered['barcode'] == barcode, ['Z(nm)', 'X(nm)', 'Y(nm)']] = np.nan

#### Calculate percentage of missing barcodes in each trace

In [10]:
missing_perc = MERFISH_data_filtered.groupby('Chromosome copy number').sum()['missing'] / n_steps * 100
missing_perc = [int(val) for val in missing_perc.values]
missing_perc_df = pd.DataFrame({'% imaged steps': missing_perc})
missing_perc_df['% imaged steps'] = 100 - missing_perc_df['% imaged steps']

#### Remove candidate outliers based on Zscore

In [11]:
MERFISH_removed_outliers = []

for trace_index, trace_i in enumerate(MERFISH_data_filtered['Chromosome copy number'].unique()):
    data_trace_i = MERFISH_data_filtered[MERFISH_data_filtered['Chromosome copy number'] == trace_i].copy()

    data_trace_i_zscore = zscore(data_trace_i[['Z(nm)', 'X(nm)', 'Y(nm)']], 0, nan_policy="omit")
    outlier_rows = np.where(abs(data_trace_i_zscore) > 3)[0].tolist()

    data_trace_i.iloc[outlier_rows, 0:3] = np.nan

    MERFISH_removed_outliers.append(data_trace_i)

    if trace_i % 1000 == 0:
            print(f"Processed up to trace {trace_i}")
    
MERFISH_removed_outliers = pd.concat(MERFISH_removed_outliers, axis=0, ignore_index=True)
MERFISH_removed_outliers.reset_index(inplace=True, drop=True)

Processed up to trace 1000
Processed up to trace 2000
Processed up to trace 3000
Processed up to trace 4000
Processed up to trace 5000
Processed up to trace 6000
Processed up to trace 7000
Processed up to trace 8000
Processed up to trace 9000
Processed up to trace 10000
Processed up to trace 11000
Processed up to trace 12000


#### Check distribution of distances between consecutive distances

In [12]:
# Convert into matrix
distances = np.zeros((len(MERFISH_removed_outliers['Chromosome copy number'].unique()), n_steps, n_steps))

for trace_index, trace_i in enumerate(MERFISH_removed_outliers['Chromosome copy number'].unique()):    
    trace_i_data = MERFISH_removed_outliers[MERFISH_removed_outliers['Chromosome copy number'] == trace_i]
    trace_i_dists_coords = trace_i_data[['X(nm)', 'Y(nm)', 'Z(nm)']].to_numpy()
    
    distances_i_mat = np.linalg.norm(trace_i_dists_coords[np.newaxis, :, :] - trace_i_dists_coords[:, np.newaxis, :], axis=2)
    distances[trace_index, :, :] = distances_i_mat

In [13]:
# Get all diagonals
all_diags = []
for i in range(0, distances.shape[0]):
    diag_i = np.diagonal(distances[i], offset=1)
    all_diags.append(diag_i)

all_diags = np.concatenate(all_diags).ravel()
all_diags = all_diags[~np.isnan(all_diags)]

print(f'Second quartile starts at {np.percentile(all_diags, 25)}')
print(f'Third quartile ends at {np.percentile(all_diags, 75)}')

Second quartile starts at 222.99327344115113
Third quartile ends at 586.6839012619998


#### Remove barcodes that are too far away from their adjacent ones

In [14]:
MERFISH_removed_outliers_extra = []

for trace_index, trace_i in enumerate(MERFISH_removed_outliers['Chromosome copy number'].unique()):    
    trace_i_data = MERFISH_removed_outliers[MERFISH_removed_outliers['Chromosome copy number'] == trace_i]
    trace_i_dists_coords = trace_i_data[['X(nm)', 'Y(nm)', 'Z(nm)']].to_numpy()

    distances_i_mat = np.linalg.norm(trace_i_dists_coords[np.newaxis, :, :] - trace_i_dists_coords[:, np.newaxis, :], axis=2)

    # Find beads spaced more than chosen distance
    distances_greater = distances_i_mat > consec_dist_thr
    distances_diag_1 = np.diag(distances_greater, k=1)
    distances_diag_2 = np.where(distances_diag_1, True, np.nan)
    distances_diag_3 = np.diff(distances_diag_2)
    # Remove first and last 
    if distances_diag_2[0] == 1:
        distances_diag_3[0] = 0
    if distances_diag_2[-1] == 1:
        distances_diag_3[-1] = 0
    bad_beads = np.where(distances_diag_3, False, True)
    bad_beads = list(np.where(bad_beads)[0] + 2)

    # Removed beads from trace
    bad_locs = np.where(trace_i_data['barcode'].isin(bad_beads))[0].tolist()
    trace_i_data.iloc[bad_locs, 0:3] = np.nan
    MERFISH_removed_outliers_extra.append(trace_i_data)
    
    if trace_i % 500 == 0:
        print(f"Processed up to trace {trace_i}")

MERFISH_removed_outliers_extra = pd.concat(MERFISH_removed_outliers_extra, axis=0, ignore_index=True)
MERFISH_removed_outliers_extra.reset_index(inplace=True, drop=True)
MERFISH_removed_outliers_extra['missing'] = (np.isnan(MERFISH_removed_outliers_extra['Z(nm)'])) & (np.isnan(MERFISH_removed_outliers_extra['X(nm)'])) & (np.isnan(MERFISH_removed_outliers_extra['Y(nm)']))

Processed up to trace 500
Processed up to trace 1000
Processed up to trace 1500
Processed up to trace 2000
Processed up to trace 2500
Processed up to trace 3000
Processed up to trace 3500
Processed up to trace 4000
Processed up to trace 4500
Processed up to trace 5000
Processed up to trace 5500
Processed up to trace 6000
Processed up to trace 6500
Processed up to trace 7000
Processed up to trace 7500
Processed up to trace 8000
Processed up to trace 8500
Processed up to trace 9000
Processed up to trace 9500
Processed up to trace 10000
Processed up to trace 10500
Processed up to trace 11000
Processed up to trace 11500
Processed up to trace 12000


#### Keep traces with > 95% steps labelled

In [15]:
good_traces_indices = np.where(MERFISH_removed_outliers_extra[['Chromosome copy number', 'missing']].groupby('Chromosome copy number').sum()['missing'] <= ((1-0.95) * n_steps))[0]

good_traces = [MERFISH_removed_outliers_extra['Chromosome copy number'].unique()[index] for index in good_traces_indices]
good_traces = MERFISH_removed_outliers_extra[MERFISH_removed_outliers_extra['Chromosome copy number'].isin(good_traces)]

In [16]:
len(good_traces['Chromosome copy number'].unique())

3188

#### Interpolate missing barcodes

In [17]:
good_traces_interpolated = good_traces.copy()
good_traces_interpolated.reset_index(inplace=True, drop=True)

for index, row in good_traces_interpolated.iterrows(): # Go through all rows
    if np.isnan(row['X(nm)']): # Identify if bead is missing
        trace = row['Chromosome copy number']
        barcode = row['barcode']

        # Find closest beads around
        trace_i_data = good_traces_interpolated[good_traces_interpolated['Chromosome copy number'] == trace]
        trace_i_data = trace_i_data.dropna(axis=0)
        trace_i_data['dist'] = (trace_i_data['barcode'] - barcode).astype(int)

        trace_i_data_left = trace_i_data[trace_i_data['dist'] < 0]
        trace_i_data_left = trace_i_data_left.copy()
        trace_i_data_left['dist_abs'] = abs(trace_i_data_left['dist']).values.tolist()
        trace_i_data_right = trace_i_data[trace_i_data['dist'] > 0]
        trace_i_data_right = trace_i_data_right.copy()
        trace_i_data_right['dist_abs'] = abs(trace_i_data_right['dist']).values.tolist()
    
        trace_i_data_close_left = trace_i_data_left.nsmallest(1, 'dist_abs')
        trace_i_data_close_right = trace_i_data_right.nsmallest(1, 'dist_abs')

        if len(trace_i_data_close_left) == 0:
            trace_i_data_close_right = trace_i_data_right.nsmallest(2, 'dist_abs')
        if len(trace_i_data_close_right) == 0:
            trace_i_data_close_left = trace_i_data_left.nsmallest(2, 'dist_abs')

        
        trace_i_data_close = pd.concat([trace_i_data_close_left, trace_i_data_close_right])
        
        # Interpolate
        bead_nums = trace_i_data_close['barcode'].values.tolist()
        x_vals = trace_i_data_close['X(nm)'].values.tolist()
        y_vals = trace_i_data_close['Y(nm)'].values.tolist()
        z_vals = trace_i_data_close['Z(nm)'].values.tolist()

        f_x = interp1d(bead_nums, x_vals, fill_value="extrapolate")
        f_y = interp1d(bead_nums, y_vals, fill_value="extrapolate")
        f_z = interp1d(bead_nums, z_vals, fill_value="extrapolate")

        x_interp = f_x(barcode)
        y_interp = f_y(barcode)
        z_interp = f_z(barcode)

        good_traces_interpolated.loc[index, ['X(nm)', 'Y(nm)', 'Z(nm)']] = [x_interp, y_interp, z_interp]

    if (row['Chromosome copy number'] % 500 == 0) & (row['barcode'] == n_steps):
        print(f"Interpolated up to trace {row['Chromosome copy number']}.")

Interpolated up to trace 1500.
Interpolated up to trace 3500.
Interpolated up to trace 4500.
Interpolated up to trace 5000.
Interpolated up to trace 5500.
Interpolated up to trace 11500.


#### Convert into matrix

In [18]:
distances_interpolated = np.zeros((len(good_traces_interpolated['Chromosome copy number'].unique()), n_steps, n_steps))

for trace_index, trace_i in enumerate(good_traces_interpolated['Chromosome copy number'].unique()):    
    trace_i_data = good_traces_interpolated[good_traces_interpolated['Chromosome copy number'] == trace_i]
    trace_i_dists_coords = trace_i_data[['X(nm)', 'Y(nm)', 'Z(nm)']].to_numpy()
    
    distances_i_mat = np.linalg.norm(trace_i_dists_coords[np.newaxis, :, :] - trace_i_dists_coords[:, np.newaxis, :], axis=2)
    distances_interpolated[trace_index, :, :] = distances_i_mat

    if trace_i % 500 == 0:
        print(f"Processed up to trace {trace_i}")

Processed up to trace 1500
Processed up to trace 3500
Processed up to trace 4500
Processed up to trace 5000
Processed up to trace 5500
Processed up to trace 11500


#### Convert into vector

In [19]:
# Extract values
distances_interpolated_df = []
for mat_num in range(distances_interpolated.shape[0]):
    curr_mat = distances_interpolated[mat_num]
    distances_interpolated_df.append(curr_mat[np.triu_indices(curr_mat.shape[0], 1)].tolist())

# Get column names
col_names = []
for bead_i in range(0, n_steps - 1):
    for bead_j in range(bead_i + 1, n_steps):
        name_i_j = f"{bead_i + 1}-{bead_j+1}"
        col_names.append(name_i_j)

distances_interpolated_df = pd.DataFrame(distances_interpolated_df, columns=col_names)

distances_interpolated_df['Chromosome copy number'] = good_traces_interpolated['Chromosome copy number'].unique().tolist()

#### Format and save

In [ ]:
distances_interpolated_df = distances_interpolated_df.iloc[:, [-1] + np.arange(0, distances_interpolated_df.shape[1]-1).tolist()]
MERFISH_data_transcription = MERFISH_data_transcription[MERFISH_data_transcription['Chromosome copy number'].isin(distances_interpolated_df['Chromosome copy number'].values.tolist())]
MERFISH_data_transcription = MERFISH_data_transcription.replace({'off;': 0, 'on;': 1})


distances_interpolated_df.to_csv('data/selected_structures_95_1000nm_thr.tsv', sep='\t', index=False)
MERFISH_data_transcription.to_csv('data/transcriptional_data_95_1000_nm_thr.tsv', sep='\t', index=False)

In [17]:
MERFISH_data_transcription

,Chromosome copy number,BACH1,CCT8,LTN1,N6AMT1,USP16
2,3,0,1,0,0,0
6,7,1,0,0,0,0
7,8,0,0,0,0,0
9,10,1,0,0,0,0
13,14,0,0,0,0,0
...,...,...,...,...,...,...
12136,12137,0,0,1,0,0
12139,12140,1,0,1,0,1
12142,12143,1,0,1,0,1
12146,12147,0,0,1,0,0


In [18]:
MERFISH_data_transcription.sum(axis=0)

Chromosome copy number    21748055
BACH1                         1296
CCT8                           668
LTN1                          1047
N6AMT1                         471
USP16                          616
dtype: object